# UROP Research Log — May 28, 2026

## Objective
Replace Nasdaq PBP data with Bloomberg-sourced data (PBP UF Equity, Cboe BZX)
and re-run the baseline tracking error analysis to confirm official numbers.

## Why
The Nasdaq data used in may26_Baseline.ipynb was flagged as an insufficiently
rigorous academic source. Today we replace it with Bloomberg Terminal data
(Source: Bloomberg L.P.) which is the industry standard for academic research.
This also allows us to incorporate NAV data alongside Market Price, enabling
a cleaner separation of expense ratio drag from synchronicity-driven noise.

## Data Sources
- PBP ETF: Bloomberg Terminal — PBP UF Equity (Cboe BZX), PX_LAST + FUND_NET_ASSET_VAL
- BXM Index: Bloomberg Terminal — BXM Index, PX_LAST
- Expense Ratio: 0.290% per annum (source: Bloomberg DES page, PBP UF Equity)

## Approach
1. Load Bloomberg CSVs, clean headers
2. Filter to 2022, merge on date
3. Re-compute log returns and daily TE
4. Compare results vs. Nasdaq-based baseline (may26)
5. Add NAV-based TE for comparison

## Known Limitations
- PBP close = 4:00 PM ET (Cboe BZX equity market close)
- BXM close = ~4:15 PM ET → Synchronicity Problem still present
- 1-min intraday data unavailable from Bloomberg (max 3 trading days for 1-min bars)
  → Stage 2 synchroniza

In [1]:
# -- Load Bloomberg CSVs --
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# --- Load PBP (Bloomberg) ---
pbp_raw = pd.read_csv("../data/raw/PBP_Bloomberg_2022.csv", skiprows=6)
pbp_raw.columns = ["date", "pbp_close", "pbp_nav"]
pbp_raw = pbp_raw.dropna(subset=["date"])
pbp_raw["date"] = pd.to_datetime(pbp_raw["date"])
pbp_raw["pbp_close"] = pd.to_numeric(pbp_raw["pbp_close"], errors="coerce")
pbp_raw["pbp_nav"] = pd.to_numeric(pbp_raw["pbp_nav"], errors="coerce")

# --- Load BXM (Bloomberg) ---
bxm_raw = pd.read_csv("../data/raw/BXM_Bloomberg_2022.csv", skiprows=6)
bxm_raw.columns = ["date", "bxm_close"]
bxm_raw = bxm_raw.dropna(subset=["date"])
bxm_raw["date"] = pd.to_datetime(bxm_raw["date"])
bxm_raw["bxm_close"] = pd.to_numeric(bxm_raw["bxm_close"], errors="coerce")

# Filter 2022
pbp_2022 = pbp_raw[pbp_raw["date"].dt.year == 2022].sort_values("date").reset_index(drop=True)
bxm_2022 = bxm_raw[bxm_raw["date"].dt.year == 2022].sort_values("date").reset_index(drop=True)

print("PBP shape:", pbp_2022.shape)
print(pbp_2022.head(3))
print("\nBXM shape:", bxm_2022.shape)
print(bxm_2022.head(3))

PBP shape: (250, 3)
        date  pbp_close  pbp_nav
0 2022-01-04      23.18  23.1783
1 2022-01-05      23.07  23.0274
2 2022-01-06        NaN  23.0419

BXM shape: (250, 2)
        date  bxm_close
0 2022-01-04    1778.63
1 2022-01-05    1767.15
2 2022-01-06    1768.30


In [2]:
# -- Merge & Compute Returns --

# Merge on date
df = pd.merge(pbp_2022, bxm_2022, on="date", how="inner")

# Drop rows where pbp_close is NaN (no market price)
df_market = df.dropna(subset=["pbp_close"]).copy()
df_market = df_market.sort_values("date").reset_index(drop=True)

# Log returns (Market Price based)
df_market["r_pbp"] = np.log(df_market["pbp_close"] / df_market["pbp_close"].shift(1))
df_market["r_bxm"] = np.log(df_market["bxm_close"] / df_market["bxm_close"].shift(1))
df_market["te_daily"] = df_market["r_pbp"] - df_market["r_bxm"]
df_market = df_market.dropna().reset_index(drop=True)

print("Merged rows (Market Price):", len(df_market))
print(df_market.head(3))

Merged rows (Market Price): 227
        date  pbp_close  pbp_nav  bxm_close     r_pbp     r_bxm  te_daily
0 2022-01-05      23.07  23.0274    1767.15 -0.004757 -0.006475  0.001719
1 2022-01-07      23.06  23.0472    1768.79 -0.000434  0.000928 -0.001361
2 2022-01-10      23.07  23.0408    1768.32  0.000434 -0.000266  0.000699


In [ ]:
# NaN check because too low merged rows count (should be 251 but its showing 227)
nan_dates = df[df["pbp_close"].isna()]["date"]
print(f"Market Price NaN counts: {len(nan_dates)}")
print(nan_dates.tolist())

Market Price NaN counts: 22
[Timestamp('2022-01-06 00:00:00'), Timestamp('2022-01-12 00:00:00'), Timestamp('2022-01-14 00:00:00'), Timestamp('2022-02-04 00:00:00'), Timestamp('2022-02-07 00:00:00'), Timestamp('2022-02-18 00:00:00'), Timestamp('2022-03-01 00:00:00'), Timestamp('2022-03-24 00:00:00'), Timestamp('2022-03-28 00:00:00'), Timestamp('2022-04-06 00:00:00'), Timestamp('2022-04-18 00:00:00'), Timestamp('2022-04-19 00:00:00'), Timestamp('2022-04-20 00:00:00'), Timestamp('2022-04-25 00:00:00'), Timestamp('2022-04-27 00:00:00'), Timestamp('2022-05-03 00:00:00'), Timestamp('2022-06-08 00:00:00'), Timestamp('2022-07-18 00:00:00'), Timestamp('2022-07-26 00:00:00'), Timestamp('2022-08-09 00:00:00'), Timestamp('2022-08-26 00:00:00'), Timestamp('2022-11-15 00:00:00')]


In [6]:
# -- Fill NaN Market Price with NAV --

df_filled = df.copy()

# Fill missing market price with NAV
df_filled["pbp_close_filled"] = df_filled["pbp_close"].fillna(df_filled["pbp_nav"])

print("NaN before fill:", df_filled["pbp_close"].isna().sum())
print("NaN after fill:", df_filled["pbp_close_filled"].isna().sum())
print(f"Days using NAV as substitute: {df_filled['pbp_close'].isna().sum()}")

# Compute log returns
df_filled = df_filled.sort_values("date").reset_index(drop=True)
df_filled["r_pbp"] = np.log(df_filled["pbp_close_filled"] / df_filled["pbp_close_filled"].shift(1))
df_filled["r_bxm"] = np.log(df_filled["bxm_close"] / df_filled["bxm_close"].shift(1))
df_filled["te_daily"] = df_filled["r_pbp"] - df_filled["r_bxm"]
df_filled = df_filled.dropna(subset=["r_pbp", "r_bxm", "te_daily"]).reset_index(drop=True)

print("Final rows:", len(df_filled))
print(df_filled.head(3))

NaN before fill: 22
NaN after fill: 0
Days using NAV as substitute: 22
Final rows: 249
        date  pbp_close  pbp_nav  bxm_close  pbp_close_filled     r_pbp  \
0 2022-01-05      23.07  23.0274    1767.15           23.0700 -0.004757   
1 2022-01-06        NaN  23.0419    1768.30           23.0419 -0.001219   
2 2022-01-07      23.06  23.0472    1768.79           23.0600  0.000785   

      r_bxm  te_daily  
0 -0.006475  0.001719  
1  0.000651 -0.001869  
2  0.000277  0.000508  


In [7]:
# -- Summary Statistics & Comparison with Nasdaq Baseline --

mean_te = df_filled["te_daily"].mean()
std_te = df_filled["te_daily"].std()
ann_te = std_te * np.sqrt(252)

# Regression
slope, intercept, r_value, p_value, std_err = stats.linregress(df_filled["r_bxm"], df_filled["r_pbp"])
r_squared = r_value ** 2
n = len(df_filled)
adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2)
t_ratio = slope / std_err

print("=== Bloomberg Baseline (Market Price + NAV fill, 2022) ===")
print(f"Observations          : {n}")
print(f"Mean Daily TE         : {mean_te:.6f}  ({mean_te*100:.4f}%)")
print(f"Std Dev Daily TE      : {std_te:.6f}  ({std_te*100:.4f}%)")
print(f"Annualized TE         : {ann_te:.6f}  ({ann_te*100:.4f}%)")
print(f"R²                    : {r_squared:.6f}")
print(f"Adj. R²               : {adj_r_squared:.6f}")
print(f"Beta                  : {slope:.6f}")
print(f"Alpha                 : {intercept:.6f}  ({intercept*100:.4f}%)")
print(f"t-ratio               : {t_ratio:.4f}")
print()
print("=== Comparison: Nasdaq vs Bloomberg ===")
print(f"{'Metric':<25} {'Nasdaq':>10} {'Bloomberg':>10}")
print(f"{'Observations':<25} {'251':>10} {n:>10}")
print(f"{'Annualized TE':<25} {'4.80%':>10} {ann_te*100:>9.2f}%")
print(f"{'R²':<25} {'0.9171':>10} {r_squared:>10.4f}")
print(f"{'Beta':<25} {'0.9826':>10} {slope:>10.4f}")
print(f"{'Alpha':<25} {'-0.0088%':>10} {intercept*100:>9.4f}%")

=== Bloomberg Baseline (Market Price + NAV fill, 2022) ===
Observations          : 249
Mean Daily TE         : -0.000065  (-0.0065%)
Std Dev Daily TE      : 0.005406  (0.5406%)
Annualized TE         : 0.085810  (8.5810%)
R²                    : 0.730946
Adj. R²               : 0.729857
Beta                  : 0.815652
Alpha                 : -0.000156  (-0.0156%)
t-ratio               : 25.9043

=== Comparison: Nasdaq vs Bloomberg ===
Metric                        Nasdaq  Bloomberg
Observations                     251        249
Annualized TE                  4.80%      8.58%
R²                            0.9171     0.7309
Beta                          0.9826     0.8157
Alpha                       -0.0088%   -0.0156%


## Thought Process — Bloomberg vs Nasdaq Data Issue

**Observation**: Bloomberg baseline gave worse results than Nasdaq:
- Annualized TE: 4.80% (Nasdaq) → 8.58% (Bloomberg)
- R²: 0.9171 (Nasdaq) → 0.7309 (Bloomberg)

**Why did this happen?**
Bloomberg had 22 missing Market Price values, which we filled with NAV.
However, PBP trades at a small premium/discount to NAV (~0.11% per DES page).
Switching between Market Price and NAV creates artificial return jumps:
- Day N: Market Price $23.07
- Day N+1: NAV $23.04 (fill) → artificial drop
- Day N+2: Market Price $23.06 → artificial rise

This NAV-fill noise inflated the tracking error artificially.

**Decision**: Use Bloomberg Market Price only (227 observations).
Drop the 22 days with missing Market Price data rather than filling with NAV.
227 observations is still statistically sufficient for academic research.

**Open Question**: Why does Bloomberg have 22 missing Market Price values
for PBP on Cboe BZX? Are these genuine non-trading days or a data gap?
Worth verifying with Professor or checking Cboe BZX trade records.

In [8]:
# -- Bloomberg Market Price Only (227 obs) --

# Use only days with actual Market Price (no NAV fill)
df_market = df.dropna(subset=["pbp_close"]).copy()
df_market = df_market.sort_values("date").reset_index(drop=True)

# Log returns
df_market["r_pbp"] = np.log(df_market["pbp_close"] / df_market["pbp_close"].shift(1))
df_market["r_bxm"] = np.log(df_market["bxm_close"] / df_market["bxm_close"].shift(1))
df_market["te_daily"] = df_market["r_pbp"] - df_market["r_bxm"]
df_market = df_market.dropna(subset=["r_pbp", "r_bxm", "te_daily"]).reset_index(drop=True)

# Stats
mean_te = df_market["te_daily"].mean()
std_te = df_market["te_daily"].std()
ann_te = std_te * np.sqrt(252)

# Regression
slope, intercept, r_value, p_value, std_err = stats.linregress(df_market["r_bxm"], df_market["r_pbp"])
r_squared = r_value ** 2
n = len(df_market)
adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2)
t_ratio = slope / std_err

print("=== Bloomberg Baseline (Market Price Only, 2022) ===")
print(f"Observations          : {n}")
print(f"Mean Daily TE         : {mean_te:.6f}  ({mean_te*100:.4f}%)")
print(f"Std Dev Daily TE      : {std_te:.6f}  ({std_te*100:.4f}%)")
print(f"Annualized TE         : {ann_te:.6f}  ({ann_te*100:.4f}%)")
print(f"R²                    : {r_squared:.6f}")
print(f"Adj. R²               : {adj_r_squared:.6f}")
print(f"Beta                  : {slope:.6f}")
print(f"Alpha                 : {intercept:.6f}  ({intercept*100:.4f}%)")
print(f"t-ratio               : {t_ratio:.4f}")
print()
print("=== Final Comparison ===")
print(f"{'Metric':<25} {'Nasdaq':>10} {'Bloomberg':>10}")
print(f"{'Observations':<25} {'251':>10} {n:>10}")
print(f"{'Annualized TE':<25} {'4.80%':>10} {ann_te*100:>9.2f}%")
print(f"{'R²':<25} {'0.9171':>10} {r_squared:>10.4f}")
print(f"{'Beta':<25} {'0.9826':>10} {slope:>10.4f}")
print(f"{'Alpha':<25} {'-0.0088%':>10} {intercept*100:>9.4f}%")

=== Bloomberg Baseline (Market Price Only, 2022) ===
Observations          : 227
Mean Daily TE         : -0.000071  (-0.0071%)
Std Dev Daily TE      : 0.005660  (0.5660%)
Annualized TE         : 0.089845  (8.9845%)
R²                    : 0.729112
Adj. R²               : 0.727909
Beta                  : 0.809759
Alpha                 : -0.000175  (-0.0175%)
t-ratio               : 24.6090

=== Final Comparison ===
Metric                        Nasdaq  Bloomberg
Observations                     251        227
Annualized TE                  4.80%      8.98%
R²                            0.9171     0.7291
Beta                          0.9826     0.8098
Alpha                       -0.0088%   -0.0175%


## Thought Process — Why Bloomberg BZX Data Performed Worse

**Observation**: Bloomberg PBP UF Equity (Cboe BZX) gave significantly worse
results than Nasdaq data:
- Annualized TE: 4.80% (Nasdaq) → 8.98% (Bloomberg BZX)
- R²: 0.9171 (Nasdaq) → 0.7291 (Bloomberg BZX)

**Root Cause**: Liquidity mismatch.
PBP UF Equity (Cboe BZX) is a secondary listing with low trading volume
(~6,326 shares/day vs ~27,579 on primary listing PBP US Equity).
Low liquidity → wider bid/ask spreads → noisier price data → inflated TE.
This is actually another form of market microstructure friction.

**Decision**: Revert to Nasdaq-sourced PBP data for now.
- Nasdaq data (251 obs, R²=0.9171) is cleaner and more representative
- The correct Bloomberg fix would be PBP US Equity (primary listing),
  not PBP UF Equity (Cboe BZX secondary listing)

**Does this affect our research conclusions?** No.
Our core finding is a relative comparison:
Standard Close TE (Before) vs. 2:55 PM Synchronized TE (After).
As long as both stages use the same PBP data source consistently,
the Before/After comparison remains valid regardless of data source.

**Next Steps**:
- [ ] Ask Professor Dodson which PBP listing to use as the official source
- [ ] If Bloomberg visit is possible again, pull PBP US Equity instead
- [ ] For now, proceed with Nasdaq data as baseline

## Results Summary (May 28, 2026)

### Final Decision: Revert to Nasdaq Baseline
After testing Bloomberg PBP UF Equity (Cboe BZX), results were significantly
worse due to low liquidity on the secondary listing. Nasdaq data remains
the working baseline until Professor Dodson advises on the correct data source.

### Confirmed Numbers (Nasdaq Baseline)
| Metric | Value |
|--------|-------|
| Observations | 251 |
| Annualized TE | 4.80% |
| R² | 0.9171 |
| Beta | 0.9826 |
| Alpha | -0.0088% |

### Bloomberg Data Acquired Today (Source: Bloomberg L.P.)
| File | Contents |
|------|----------|
| PBP_Bloomberg_2022.csv | PX_LAST + NAV (PBP UF Equity) |
| BXM_Bloomberg_2022.csv | PX_LAST |
| SPX_Bloomberg_2022.csv | PX_LAST + OHLC |
| VIX_Bloomberg_2022.csv | PX_LAST |
| BXMD_Bloomberg_2022.csv | PX_LAST |

### Key Info from Bloomberg DES
- Expense Ratio: **0.290%** per annum
- 1Y Price Tracking Error: 1.334% (Bloomberg official)
- 1Y NAV Tracking Error: 0.635% (Bloomberg official)

## Next Steps
- [ ] Ask Professor Dodson: correct PBP data source (PBP US vs PBP UF)?
- [ ] Begin Cboe tick data EDA (may28_Cboe_EDA.ipynb)
- [ ] Stage 2: 3:55 PM synchronization pipeline